In [4]:
import numpy as np
import time

In [5]:
def compute_dft_manual(signal):
    signal = np.asarray(signal, dtype=complex)
    N = signal.shape[0]

    if N == 0:
        return np.array([])

    n = np.arange(N)

    k = n.reshape((N, 1))

    # Create the DFT Matrix (Twiddle Factor Matrix)
    # W_kn = exp(-2j * pi * k * n / N)

    # (N, 1) * (1, N) -> (N, N) matrix
    kn_matrix = k * n

    # 2. Calculate the full exponent matrix M
    M = -2j * np.pi * kn_matrix / N

    # 3. Compute the DFT matrix W
    W = np.exp(M)

    dft_coefficients = np.dot(W, signal)

    return dft_coefficients

In [6]:
def get_frequencies_manual(n, sample_spacing=1.0):
    if n <= 0:
        return np.array([])

    # sample_rate = 1 / sample_spacing
    # The fundamental frequency resolution is sample_rate / n
    freq_resolution = (1.0 / sample_spacing) / n

    k_indices = np.arange(n)

    midpoint = (n + 1) // 2

    k_indices[midpoint:] = k_indices[midpoint:] - n

    frequencies = k_indices * freq_resolution

    return frequencies

In [7]:
sample_rate = 100  # Hz (samples per second)
duration = 2  # seconds
N = sample_rate * duration  # Total number of samples

# Create a time vector from 0 to 'duration'
# 'endpoint=False' is important for FFT analysis
t = np.linspace(0.0, duration, N, endpoint=False)

# Signal components
freq1 = 5  # 5 Hz
freq2 = 12  # 12 Hz
signal = 0.8 * np.sin(2 * np.pi * freq1 * t) + 0.5 * np.sin(2 * np.pi * freq2 * t)

print(f"Generated a signal with {N} samples.")
print(f"Signal contains frequencies: {freq1} Hz and {freq2} Hz.\n")

start_time_manual = time.time()
dft_values = compute_dft_manual(signal)
end_time_manual = time.time()
print(f"Custom DFT took: {end_time_manual - start_time_manual:.6f} seconds\n")

# The sample spacing is 1 / sample_rate
frequencies = get_frequencies_manual(N, sample_spacing=1.0 / sample_rate)

amplitudes = 2.0 / N * np.abs(dft_values)

positive_freq_indices = np.where(frequencies >= 0)
positive_frequencies = frequencies[positive_freq_indices]
positive_amplitudes = amplitudes[positive_freq_indices]

peak_indices = np.argsort(positive_amplitudes)[-5:]  # Get top 5 indices

start_time_fft = time.time()
fft_values = np.fft.fft(signal)
end_time_fft = time.time()
print(f"NumPy FFT took: {end_time_fft - start_time_fft:.6f} seconds")

fft_frequencies = np.fft.fftfreq(N, d=1.0 / sample_rate)

all_close_values = np.allclose(dft_values, fft_values)
all_close_freqs = np.allclose(frequencies, fft_frequencies)

print(f"\nCustom DFT values match np.fft.fft():  {all_close_values}")
print(f"Custom freqs match np.fft.fftfreq(): {all_close_freqs}")

Generated a signal with 200 samples.
Signal contains frequencies: 5 Hz and 12 Hz.

Custom DFT took: 0.017404 seconds

NumPy FFT took: 0.011096 seconds

Custom DFT values match np.fft.fft():  True
Custom freqs match np.fft.fftfreq(): True
